# Amazon Bedrock AgentCore Payments 설정

## 개요

AI agent용 payment infrastructure를 구축하려면 안전한 wallet 관리, 결정론적 payment limit 적용, 변화하는 protocol 전반의 payment orchestration을 한꺼번에 해결해야 합니다. agent level에서는 문제가 더 복잡해집니다. 하나의 task가 서로 다른 결제 요구 사항을 가진 수십 개의 x402 endpoint를 호출할 수 있고, 에이전트는 다음 호출 대상을 비결정론적으로 판단합니다.

**Amazon Bedrock AgentCore payments**가 이 모든 작업을 처리합니다. 이 튜토리얼에서는 이후 모든 튜토리얼이 사용하는 전체 payment stack을 설정합니다.

> **비용 안내:** 이 튜토리얼에서는 현실 세계의 가치가 없는 testnet USDC를 사용하지만 AWS infrastructure(AgentCore payments API 호출, Secrets Manager storage, CloudWatch Logs)에는 표준 AWS 요금이 발생합니다. persistent resource는 정리할 때까지 계속 비용이 발생합니다. 하단의 리소스 정리 섹션을 참조하세요.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                            |
|:--------------------|:---------------------------------------------------------------------|
| 튜토리얼 유형       | Task 기반                                                            |
| 튜토리얼 구성 요소  | IAM role, Payment Manager, Connector, Embedded Wallet Instrument, Session |
| 예제 난이도         | 쉬움                                                                 |
| 사용 SDK            | boto3                                                                |

### API Reference

| Operation | Plane | Client | 설명 |
|:----------|:------|:-------|:------------|
| `CreatePaymentCredentialProvider` | Control | `bedrock-agentcore-control` | wallet credentials를 안전하게 저장 |
| `CreatePaymentManager` | Control | `bedrock-agentcore-control` | top-level payment 구성 |
| `CreatePaymentConnector` | Control | `bedrock-agentcore-control` | Manager를 Credential Provider에 연결 |
| `CreatePaymentInstrument` | Data | `bedrock-agentcore` | crypto wallet provision |
| `CreatePaymentSession` | Data | `bedrock-agentcore` | 시간 제한 spending authorization |

> **Testnet 전용입니다.** 모든 코드는 [faucet.circle.com](https://faucet.circle.com/)에서 무료 USDC를 받아 Base Sepolia 또는 Solana Devnet을 사용합니다. Testnet USDC는 현실 세계의 가치가 없습니다.

> **AWS 리소스 비용:** testnet USDC는 무료지만 AWS 리소스(Payment Manager, CloudWatch Logs, X-Ray trace)에는 요금이 발생합니다. 24시간 이내에 정리하면 예상 비용은 $1 미만입니다. 마지막의 리소스 정리 섹션을 참조하세요.

### 지원 조합

AgentCore payments는 provider 및 network에 종속되지 않습니다. 원하는 조합을 선택하세요.

| Provider | Network | Instrument Type | Faucet |
|----------|---------|-----------------|--------|
| Coinbase CDP | Base (ETHEREUM) | `EMBEDDED_CRYPTO_WALLET` | faucet.circle.com → Base Sepolia |
| Coinbase CDP | Solana (SOLANA) | `EMBEDDED_CRYPTO_WALLET` | faucet.circle.com → Solana Devnet |
| Stripe (Privy) | Base (ETHEREUM) | `EMBEDDED_CRYPTO_WALLET` | faucet.circle.com → Base Sepolia |
| Stripe (Privy) | Solana (SOLANA) | `EMBEDDED_CRYPTO_WALLET` | faucet.circle.com → Solana Devnet |

모든 조합은 사용자 identity에 `linkedAccounts`를 사용하는 **embedded wallet**(`EMBEDDED_CRYPTO_WALLET`)을 사용합니다. `.env`에서 `CREDENTIAL_PROVIDER_TYPE`과 `NETWORK`를 설정하여 선택합니다. 이후 튜토리얼의 agent 코드는 어떤 조합을 선택해도 동일합니다.

## 아키텍처

```
┌───────────────────────────────────────────────────────────┐
│                    개발자 / 관리자                        │
│                 (ControlPlaneRole)                        │
│                                                           │
│  CredentialProvider → PaymentManager → PaymentConnector   │
│  한 번 설정하여 payment stack 생성                        │
└─────────────────────────────┬─────────────────────────────┘
                              │
                              ▼
┌───────────────────────────────────────────────────────────┐
│             Application Backend                           │
│                 (ManagementRole)                          │
│                                                           │
│  CreateInstrument(wallet) → CreateSession(budget)          │
│  ProcessPayment를 호출할 수 없음                          │
└─────────────────────────┬─────────────────────────────────┘
                          │ sessionId + instrumentId 전달
                          ▼
┌───────────────────────────────────────────────────────────┐
│                    Agent Runtime                          │
│               (ProcessPaymentRole)                        │
│                                                           │
│  ProcessPayment 전용. session/instrument 생성 불가        │
└───────────────────────────────────────────────────────────┘
```

**ResourceRetrievalRole**은 Runtime에서 AgentCore가 assume하는 service role이며 직접 호출하지 않습니다.

### Role 분리

![Role 분리](images/role_separation.png)


## 사전 요구 사항

* Python 3.10+ 및 Jupyter
* AWS CLI 구성 완료(`aws sts get-caller-identity`로 검증)
* AgentCore payments 액세스 권한이 있는 AWS 계정
* Wallet provider credentials
  - **Coinbase CDP:** [providers/coinbase_cdp_account_setup.ipynb](providers/coinbase_cdp_account_setup.ipynb) 참조
  - **Stripe(Privy):** [providers/stripe_privy_account_setup.ipynb](providers/stripe_privy_account_setup.ipynb) 참조
* `.env` 구성: `cp .env.sample .env` 실행 후 값 입력


In [ ]:
%pip install -r requirements.txt --quiet

## Region 선택

아래에서 `AWS_REGION`을 설정합니다. 모든 리소스(Payment Manager, Connector, Instrument, Session)가 이 region에 생성됩니다. 이후 튜토리얼은 `.env`에서 이 값을 가져옵니다.

In [ ]:
import os
import boto3

# ✏️ 여기에 region 설정 — 변경해야 하는 유일한 위치
AWS_REGION = "us-west-2"

os.environ["AWS_REGION"] = AWS_REGION

# ✏️ 이름이 지정된 AWS profile을 사용하려면 주석 해제
# os.environ['AWS_PROFILE'] = '<your-profile>'

session = boto3.Session(region_name=AWS_REGION)
identity = session.client("sts").get_caller_identity()
print(f"Region:  {AWS_REGION}")
print(f"Account: {identity['Account']}")
print(f"Identity: {identity['Arn']}")

## 0a단계 — IAM Role 생성

튜토리얼용 IAM role 네 개를 생성합니다. role이 이미 있으면 이 단계에서 업데이트합니다.

| Role | 권한 | 용도 |
|:-----|:-----------|:--------|
| **ControlPlaneRole** | Manager, Connector, Credential Provider 생성/관리 | Admin 설정 |
| **ManagementRole** | Instrument/Session CRUD, ProcessPayment **Deny** | App backend |
| **ProcessPaymentRole** | `ProcessPayment` + read query | Agent 실행 |
| **ResourceRetrievalRole** | Secrets Manager, token-vault, `sts:SetContext` | Service role(내부) |

### Role을 분리하는 이유

role 분리는 application backend와 Runtime process 사이에 보안 경계를 적용합니다. application backend(ManagementRole)는 spending limit이 있는 instrument와 session을 생성합니다. Runtime process(ProcessPaymentRole)는 에이전트를 대신하여 `ProcessPayment`를 호출하는 결정론적 코드를 실행하며, 에이전트(LLM)는 `ProcessPayment`를 직접 호출하지 않습니다. Runtime은 session을 생성하거나 limit을 override하거나 새 wallet을 provision할 수 없습니다. 이를 통해 budget을 설정하는 코드와 payment를 처리하는 코드가 구조적으로 분리되어 budget control이 유지됩니다.

같은 이유로 ManagementRole에는 `ProcessPayment`에 대한 명시적 Deny가 있습니다. budget을 설정하는 코드는 payment를 처리해서는 안 됩니다. 이는 application logic이 아니라 IAM으로 적용되는 구조적 제약입니다. role에 관한 자세한 내용은 [공식 AWS 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-iam-roles.html#payments-iam-why-separation)를 참조하세요.


In [ ]:
import sys

sys.path.append("..")
from utils import setup_payment_roles

roles = setup_payment_roles()

# 이후 셀에서 사용할 role ARN 저장
CONTROL_PLANE_ROLE_ARN = roles["control_plane"]
MANAGEMENT_ROLE_ARN = roles["management"]
PROCESS_PAYMENT_ROLE_ARN = roles["process_payment"]
RESOURCE_RETRIEVAL_ROLE_ARN = roles["resource_retrieval"]

# 이후 튜토리얼을 위해 모든 role ARN을 .env에 저장
from utils import update_env_file

update_env_file(
    {
        "CONTROL_PLANE_ROLE_ARN": CONTROL_PLANE_ROLE_ARN,
        "MANAGEMENT_ROLE_ARN": MANAGEMENT_ROLE_ARN,
        "PROCESS_PAYMENT_ROLE_ARN": PROCESS_PAYMENT_ROLE_ARN,
        "PROCESS_PAYMENT_ROLE_NAME": PROCESS_PAYMENT_ROLE_ARN.split("/")[-1],
        "RESOURCE_RETRIEVAL_ROLE_ARN": RESOURCE_RETRIEVAL_ROLE_ARN,
    }
)

print(f"\n  ControlPlaneRole:      {CONTROL_PLANE_ROLE_ARN}")
print(f"  ManagementRole:        {MANAGEMENT_ROLE_ARN}")
print(f"  ProcessPaymentRole:    {PROCESS_PAYMENT_ROLE_ARN}")
print(f"  ResourceRetrievalRole: {RESOURCE_RETRIEVAL_ROLE_ARN}")

## 1단계 — 환경 구성

`.env`에서 설정을 load하고 현재 구성을 검증합니다. 최종 사용자 wallet을 소유한 identity인 `LINKED_EMAIL`이 아직 설정되지 않았다면 이 셀에서 입력을 요청하고 `.env`에 저장합니다.


In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)
os.environ["AWS_REGION"] = AWS_REGION

PAYMENTS_CP_ENDPOINT = f"https://bedrock-agentcore-control.{AWS_REGION}.amazonaws.com"
PAYMENTS_DP_ENDPOINT = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com"
CREDENTIAL_PROVIDER_ENDPOINT = PAYMENTS_CP_ENDPOINT

CREDENTIAL_PROVIDER_TYPE = os.environ.get("CREDENTIAL_PROVIDER_TYPE", "CoinbaseCDP")

MANAGER_NAME = os.environ.get("DEFAULT_PAYMENT_MANAGER_NAME", "MyPaymentManager")
_derived_connector_name = {
    "CoinbaseCDP": "MyCoinbaseConnector",
    "StripePrivy": "MyPrivyConnector",
}.get(CREDENTIAL_PROVIDER_TYPE, "MyPaymentConnector")
CONNECTOR_NAME = os.environ.get("DEFAULT_PAYMENT_CONNECTOR_NAME") or _derived_connector_name
USER_ID = os.environ.get("USER_ID", "test-user-001")
NETWORK = os.environ.get("NETWORK", "ETHEREUM")

# .env에서 role ARN load(0a단계에서 저장됨)
CONTROL_PLANE_ROLE_ARN = os.environ.get("CONTROL_PLANE_ROLE_ARN", "")
MANAGEMENT_ROLE_ARN = os.environ.get("MANAGEMENT_ROLE_ARN", "")
PROCESS_PAYMENT_ROLE_ARN = os.environ.get("PROCESS_PAYMENT_ROLE_ARN", "")
RESOURCE_RETRIEVAL_ROLE_ARN = os.environ.get("RESOURCE_RETRIEVAL_ROLE_ARN", "")

LINKED_EMAIL = os.environ.get("LINKED_EMAIL", "").strip()
if not LINKED_EMAIL or LINKED_EMAIL.startswith("<") or LINKED_EMAIL == "user@example.com":
    print("LINKED_EMAIL is not set yet. Enter a real email address you can receive mail at")
    print("(used for wallet hub login and Privy OTP codes).")
    LINKED_EMAIL = input("Email: ").strip()
    if "@" not in LINKED_EMAIL or LINKED_EMAIL.startswith("<"):
        raise ValueError("A valid email address is required. Re-run this cell or edit .env directly.")
    from utils import update_env_file

    update_env_file({"LINKED_EMAIL": LINKED_EMAIL})
    os.environ["LINKED_EMAIL"] = LINKED_EMAIL


def _check(label, value, redact=False):
    ok = value and not value.startswith("<")
    icon = "✅" if ok else "❌ MISSING"
    display = "[redacted]" if redact and value else value
    print(f"  {icon}  {label}: {display}")


print(f"\n  Provider: {CREDENTIAL_PROVIDER_TYPE}")
print("\n  AWS:")
_check("AWS_REGION", AWS_REGION)
_check("CP_ENDPOINT", PAYMENTS_CP_ENDPOINT)
_check("DP_ENDPOINT", PAYMENTS_DP_ENDPOINT)
print("\n  IAM Roles:")
_check("CONTROL_PLANE_ROLE_ARN", CONTROL_PLANE_ROLE_ARN)
_check("MANAGEMENT_ROLE_ARN", MANAGEMENT_ROLE_ARN)
_check("PROCESS_PAYMENT_ROLE_ARN", PROCESS_PAYMENT_ROLE_ARN)
_check("RESOURCE_RETRIEVAL_ROLE_ARN", RESOURCE_RETRIEVAL_ROLE_ARN)
print("\n  Identity:")
_check("LINKED_EMAIL", LINKED_EMAIL)

## 2단계 — IAM Role 검증

role은 0a단계에서 생성했습니다. 이 셀에서 존재 여부를 확인합니다.

In [ ]:
import botocore.exceptions
from utils import (
    CONTROL_PLANE_ROLE,
    MANAGEMENT_ROLE,
    PROCESS_PAYMENT_ROLE,
    RESOURCE_RETRIEVAL_ROLE,
)

session = boto3.Session(region_name=AWS_REGION)
iam = session.client("iam")

for name in [
    CONTROL_PLANE_ROLE,
    MANAGEMENT_ROLE,
    PROCESS_PAYMENT_ROLE,
    RESOURCE_RETRIEVAL_ROLE,
]:
    try:
        arn = iam.get_role(RoleName=name)["Role"]["Arn"]
        print(f"  ✅ {name}")
    except botocore.exceptions.ClientError:
        print(f"  ❌ {name} — not found. Re-run Step 0a.")
        raise

print("\n✅ All 4 IAM roles present.")

## 3단계 — boto3 Client 생성

AgentCore payments는 별도의 API endpoint 두 개를 사용합니다.

* **Control Plane**(`bedrock-agentcore-control`) — payment stack(Manager, Connector, Credential Provider) 관리
* **Data Plane**(`bedrock-agentcore`) — payment operation(Instrument, Session, ProcessPayment) 실행

아래 셀에서는 각 작업에 적절한 IAM role을 assume합니다.
* `ControlPlaneRole` → `cp_client` — 4~6단계
* `ManagementRole` → `dp_client` — 7~8단계

In [ ]:
from utils import assume_role

print("Assuming ControlPlaneRole...")
cp_session = assume_role(session, CONTROL_PLANE_ROLE_ARN, "tutorial-00-cp")
cp_client = cp_session.client("bedrock-agentcore-control", endpoint_url=PAYMENTS_CP_ENDPOINT)
cred_client = cp_session.client("bedrock-agentcore-control", endpoint_url=CREDENTIAL_PROVIDER_ENDPOINT)

print("\nAssuming ManagementRole...")
mgmt_session = assume_role(session, MANAGEMENT_ROLE_ARN, "tutorial-00-mgmt")
dp_client = mgmt_session.client("bedrock-agentcore", endpoint_url=PAYMENTS_DP_ENDPOINT)

print("\n✅ Clients ready")

## 4단계 — Credential Provider 생성

**PaymentCredentialProvider**는 wallet provider credentials(Coinbase API key 또는 Privy key)를 AgentCore Identity 내부에 안전하게 저장합니다. credentials를 ingest한 후에는 코드에 반환하지 않으며, service가 Runtime에서 ResourceRetrievalRole을 통해 가져옵니다.

wallet provider가 key material을 보유합니다. AgentCore는 Runtime에서 ResourceRetrievalRole을 통해 credentials를 가져오며, agent 코드는 private key를 직접 처리하지 않습니다.

In [ ]:
import uuid
from utils import pp, idempotent_create, client_token, require_env

CRED_PROVIDER_NAME = f"{CREDENTIAL_PROVIDER_TYPE}{uuid.uuid4().hex[:8]}"

if CREDENTIAL_PROVIDER_TYPE == "CoinbaseCDP":
    provider_config = {
        "coinbaseCdpConfiguration": {
            "apiKeyId": require_env("COINBASE_API_KEY_ID"),
            "apiKeySecret": require_env("COINBASE_API_KEY_SECRET"),
            "walletSecret": require_env("COINBASE_WALLET_SECRET"),
        }
    }
elif CREDENTIAL_PROVIDER_TYPE == "StripePrivy":
    provider_config = {
        "stripePrivyConfiguration": {
            "appId": require_env("PRIVY_APP_ID"),
            "appSecret": require_env("PRIVY_APP_SECRET"),
            "authorizationId": require_env("PRIVY_AUTHORIZATION_ID"),
            "authorizationPrivateKey": require_env("PRIVY_AUTHORIZATION_PRIVATE_KEY"),
        }
    }
else:
    raise ValueError(f"Unknown CREDENTIAL_PROVIDER_TYPE: {CREDENTIAL_PROVIDER_TYPE}")

resp = idempotent_create(
    cred_client.create_payment_credential_provider,
    f"Credential provider '{CRED_PROVIDER_NAME}' already exists",
    name=CRED_PROVIDER_NAME,
    credentialProviderVendor=CREDENTIAL_PROVIDER_TYPE,
    providerConfigurationInput=provider_config,
)
if resp:
    pp("CreatePaymentCredentialProvider", resp)
    CREDENTIAL_PROVIDER_ARN = resp["credentialProviderArn"]
    print(f"\n  credentialProviderArn: {CREDENTIAL_PROVIDER_ARN}")

## 4b단계 — Secret Access 제한(Security Best Practice)

설정 후에는 ResourceRetrievalRole만 읽을 수 있도록 Secrets Manager의 credential provider secret을 제한하는 것이 좋습니다. [Secrets Manager console](https://console.aws.amazon.com/secretsmanager/)에서 prefix가 `bedrock-agentcore-identity`인 secret을 찾아 `ResourceRetrievalRole`을 제외한 모든 principal에 `GetSecretValue`를 거부하는 resource policy를 추가합니다.

전체 policy template은 [문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-iam-roles.html)를, 자세한 내용은 [Configure credential provider](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/resource-providers.html)를 참조하세요.

## 5단계 — Payment Manager 생성

top-level resource입니다. Runtime에서 credentials에 액세스할 때 ResourceRetrievalRole을 사용합니다.

**참고:** idempotency를 위해 `clientToken`은 33 chars 이상이어야 합니다.

In [ ]:
from utils import wait_for_status

resp = idempotent_create(
    cp_client.create_payment_manager,
    f"Manager '{MANAGER_NAME}' already exists",
    name=MANAGER_NAME,
    description=f"{MANAGER_NAME} Tutorial 00",
    authorizerType="AWS_IAM",
    roleArn=RESOURCE_RETRIEVAL_ROLE_ARN,
    clientToken=client_token(),
)
if resp:
    pp("CreatePaymentManager", resp)
    MANAGER_ID = resp["paymentManagerId"]
    MANAGER_ARN = resp["paymentManagerArn"]
    print(f"\n  ID (control plane): {MANAGER_ID}")
    print(f"  ARN (data plane):   {MANAGER_ARN}")

    print("\nWaiting for READY...")
    wait_for_status(cp_client.get_payment_manager, "READY", paymentManagerId=MANAGER_ID)
    print("✅ PaymentManager is READY")

    # 이후 셀과 재실행 시 항상 올바른 ID를 사용하도록 즉시 저장
    from utils import update_env_file

    update_env_file(
        {
            "AWS_REGION": AWS_REGION,
            "PAYMENT_MANAGER_ARN": MANAGER_ARN,
            "PAYMENT_MANAGER_ID": MANAGER_ID,
            "CREDENTIAL_PROVIDER_ARN": CREDENTIAL_PROVIDER_ARN,
            "CREDENTIAL_PROVIDER_TYPE": CREDENTIAL_PROVIDER_TYPE,
            "USER_ID": USER_ID,
            "NETWORK": NETWORK,
        }
    )

## 6단계 — Payment Connector 생성

Manager를 Credential Provider에 연결합니다.

In [ ]:
connector_type = "CoinbaseCDP" if CREDENTIAL_PROVIDER_TYPE == "CoinbaseCDP" else "StripePrivy"
cred_key = "coinbaseCDP" if CREDENTIAL_PROVIDER_TYPE == "CoinbaseCDP" else "stripePrivy"

resp = idempotent_create(
    cp_client.create_payment_connector,
    f"Connector '{CONNECTOR_NAME}' already exists",
    paymentManagerId=MANAGER_ID,
    name=CONNECTOR_NAME,
    description=f"{CONNECTOR_NAME} {connector_type}",
    type=connector_type,
    credentialProviderConfigurations=[{cred_key: {"credentialProviderArn": CREDENTIAL_PROVIDER_ARN}}],
    clientToken=client_token(),
)
if resp:
    pp("CreatePaymentConnector", resp)
    CONNECTOR_ID = resp["paymentConnectorId"]
    print(f"\n  paymentConnectorId: {CONNECTOR_ID}")

    print("\nWaiting for READY...")
    wait_for_status(
        cp_client.get_payment_connector,
        "READY",
        paymentManagerId=MANAGER_ID,
        paymentConnectorId=CONNECTOR_ID,
    )
    print("✅ PaymentConnector is READY")

    # 즉시 저장
    from utils import update_env_file

    update_env_file({"PAYMENT_CONNECTOR_ID": CONNECTOR_ID})

## 7단계 — Payment Instrument 생성(Embedded Wallet)

사용자 identity에 연결된 embedded USDC wallet을 provision합니다. 모든 data plane 호출에서는 short ID가 아니라 `paymentManagerArn`(전체 ARN)을 사용합니다.

Embedded wallet은 `linkedAccounts`와 함께 `EMBEDDED_CRYPTO_WALLET`을 사용하여 wallet을 사용자 email에 연결합니다. Coinbase CDP와 Stripe/Privy 모두 동일합니다.

**참고:** `userId`는 HTTP header `X-Amzn-Bedrock-AgentCore-Payments-User-Id`로 전송됩니다. USDC 금액은 소수점 6자리를 사용합니다: `100000` = $0.10.

In [ ]:
resp = dp_client.create_payment_instrument(
    paymentManagerArn=MANAGER_ARN,
    paymentConnectorId=CONNECTOR_ID,
    userId=USER_ID,
    paymentInstrumentType="EMBEDDED_CRYPTO_WALLET",
    paymentInstrumentDetails={
        "embeddedCryptoWallet": {
            "network": NETWORK,
            "linkedAccounts": [{"email": {"emailAddress": LINKED_EMAIL}}],
        }
    },
    clientToken=client_token(),
)
pp("CreatePaymentInstrument", resp)
INSTRUMENT_ID = resp["paymentInstrument"]["paymentInstrumentId"]
WALLET_ADDRESS = resp["paymentInstrument"]["paymentInstrumentDetails"]["embeddedCryptoWallet"]["walletAddress"]
print(f"\n  instrumentId:  {INSTRUMENT_ID}")
print(f"  walletAddress: {WALLET_ADDRESS}")

print("\nWaiting for ACTIVE...")
wait_for_status(
    dp_client.get_payment_instrument,
    "ACTIVE",
    paymentManagerArn=MANAGER_ARN,
    paymentConnectorId=CONNECTOR_ID,
    paymentInstrumentId=INSTRUMENT_ID,
    userId=USER_ID,
)
print("✅ Instrument is ACTIVE")

# 즉시 저장
from utils import update_env_file

update_env_file({"INSTRUMENT_ID": INSTRUMENT_ID, "WALLET_ADDRESS": WALLET_ADDRESS})

## 7a단계 — WalletHub URL 가져오기(Coinbase 전용)

Coinbase embedded wallet에는 signing 권한을 부여하고 on-ramp funding option을 확인할 수 있도록 browser에서 여는 hosted **WalletHub** URL이 제공됩니다. URL은 instrument가 `ACTIVE`가 된 후 asynchronous 방식으로 provision되므로 이 셀에서 잠시 polling합니다.

Privy 사용자는 건너뛸 수 있습니다. 이에 해당하는 hosted 기능이 없습니다. signing consent는 7b단계에서 설명하는 Privy reference frontend를 통해 진행합니다.

In [ ]:
# WalletHub URL 가져오기(CoinbaseCDP 전용)
# redirectUrl은 asynchronous 방식으로 provision되며 ACTIVE 이후 몇 초 걸릴 수 있음
# 처음에 표시되지 않으면 이 셀을 다시 실행
#
# 참고: boto3 service model의 response shape에 redirectUrl이 포함되지 않을 수 있으므로
# 여기서는 boto3 dp_client 대신 PaymentManager SDK를 사용함
if CREDENTIAL_PROVIDER_TYPE != "CoinbaseCDP":
    print(f"  Skipping WalletHub fetch ({CREDENTIAL_PROVIDER_TYPE} uses the Privy reference frontend).")
else:
    import time
    from bedrock_agentcore.payments import PaymentManager

    pm = PaymentManager(payment_manager_arn=MANAGER_ARN, region_name=AWS_REGION)

    redirect_url = None
    for attempt in range(6):
        instr_details = pm.get_payment_instrument(
            user_id=USER_ID,
            payment_instrument_id=INSTRUMENT_ID,
        )
        wallet_info = instr_details.get("paymentInstrumentDetails", {}).get("embeddedCryptoWallet", {})
        redirect_url = wallet_info.get("redirectUrl")
        if redirect_url:
            break
        if attempt < 5:
            time.sleep(5)

    if redirect_url:
        print(f"  WalletHub: {redirect_url}")
        print("  Open this URL to fund the wallet and grant signing permission.")
    else:
        print("  ⚠️  WalletHub URL not yet available after ~25s.")
        print("     This can happen on newly-created instruments — re-run this cell.")

## 7b단계 — Wallet 자금 입금 + Signing 위임

Tutorial 00을 실행하기 전에 Delegated Signing을 활성화하세요. ProcessPayment에는 이 구성이 필요합니다.

에이전트가 이 wallet에서 x402 payment를 생성하려면 두 가지 작업이 필요합니다.

1. testnet USDC로 **wallet에 자금을 입금**합니다.
2. **signing 위임** — 최종 사용자가 자신의 wallet에서 sign할 권한을 에이전트에 부여합니다.

> **두 identity가 사용됩니다.** 이 튜토리얼에서는 두 역할을 모두 수행합니다. *developer*는 AWS credentials를 사용하여 AgentCore API를 호출하고, *최종 사용자*는 `LINKED_EMAIL` identity로 wallet hub에 로그인하여 signing 권한을 부여합니다. Notebook 셀에는 AWS credentials를 사용하고 Coinbase WalletHub 또는 Privy reference frontend에 로그인할 때는 `LINKED_EMAIL`을 사용하세요.

ProcessPayment가 성공하려면 Delegated signing이 필요합니다. 계속 진행하기 전에 signing 권한을 부여하세요.

### x402 Payment Flow

![x402 Payment Flow](images/x402_payment_flow.png)


### 1. Wallet에 자금 입금

faucet에서 **20 USDC**를 요청합니다. 모든 튜토리얼에 충분한 금액입니다.
- 각 x402 payment는 일반적으로 USDC $0.01~$0.10
- 튜토리얼 session은 $0.20~$2.00의 budget 사용
- 튜토리얼별 실제 지출은 약 $0.05~$0.50(대부분 micro-transaction)
- 20 USDC면 8개 튜토리얼을 모두 수행하고도 충분함

아래 셀에 출력된 wallet address를 복사한 후 다음을 수행합니다.

1. **[faucet.circle.com](https://faucet.circle.com/)**으로 이동합니다.
2. `NETWORK` 설정과 일치하는 network를 선택합니다.
   - `ETHEREUM` → **Base Sepolia**
   - `SOLANA` → **Solana Devnet**
3. address를 붙여넣고 USDC를 요청합니다.


In [ ]:
print(f"\n  Wallet: {WALLET_ADDRESS}")
print(f"  Network: {NETWORK}")
faucet_network = "Base Sepolia" if NETWORK == "ETHEREUM" else "Solana Devnet"
print(f"  Faucet: https://faucet.circle.com/ → select {faucet_network}")
if NETWORK == "ETHEREUM":
    print(f"  Verify: https://sepolia.basescan.org/address/{WALLET_ADDRESS}")
else:
    print(f"  Verify: https://explorer.solana.com/address/{WALLET_ADDRESS}?cluster=devnet")
print("\n  ✋ ACTION: Fund the wallet with 20 USDC before continuing.")
print("     20 USDC covers all tutorials (typical spend: $0.05–$0.50 per tutorial).")

### 2. Signing 위임

flow는 provider에 따라 다릅니다.

#### CoinbaseCDP

Coinbase에서는 **WalletHub**(위 7단계에서 출력된 `redirectUrl`)가 signing 권한을 처리합니다. 해당 URL을 열고 `LINKED_EMAIL`로 로그인합니다.

- **signing 권한 부여** — WalletHub에서 에이전트의 delegated signing 승인을 요청합니다. 여기서 필요한 작업은 이것뿐입니다.
- **Funding** — 위 단계에서 Circle faucet을 통해 이미 자금을 입금했습니다. WalletHub에는 on-ramp option(Credit/Debit, Google Pay, Apple Pay, ACH)도 표시되지만 이는 testnet이 아닌 실제 USDC용입니다. 이 튜토리얼에는 faucet만 필요합니다.

권한을 부여하면 이 CDP project 아래의 모든 embedded wallet에 delegation이 적용됩니다.

> **검증:** [CDP Portal](https://portal.cdp.coinbase.com/) → **Wallets** → **Embedded Wallet** → **Policies** tab을 열고 **Delegated Signing**이 활성화되었는지 확인합니다.

#### StripePrivy

1. `providers/stripe_privy_account_setup.ipynb`의 6단계에서 시작한 Privy reference frontend가 이 작업을 처리합니다.
2. browser에서 [http://localhost:3000](http://localhost:3000)을 열고 `.env`의 `LINKED_EMAIL`에 사용한 **동일한 email**로 로그인합니다.
3. **wallet과 balance를 확인합니다.** Privy reference frontend에서 wallet address가 위 7단계에 출력된 `WALLET_ADDRESS`와 일치하고 balance에 Circle faucet에서 입금한 USDC가 반영되었는지 확인합니다. address 또는 balance가 잘못되었다면 다른 사용자로 로그인한 것입니다. 로그아웃한 후 `.env`의 `LINKED_EMAIL`로 다시 로그인하고 계속 진행하세요.
4. **Connect agent → Give access**를 선택합니다. 그러면 사용자의 각 Privy wallet에 AgentCore가 *additional signer*로 추가됩니다.
5. **이 작업은 사용자가 보유한 모든 Privy wallet에 signer access를 부여합니다.** 이 Notebook을 `NETWORK=SOLANA`로 실행했다면 위에서 Solana wallet이 최근 provision되었습니다. Privy의 새 wallet이 페이지에 반영되도록 browser에서 Privy reference frontend를 다시 load한 후 Connect agent를 한 번 더 선택합니다.

Privy reference frontend가 실행 중이 아니라면 `providers/stripe_privy_account_setup.ipynb`의 6단계로 돌아가 시작합니다.

In [ ]:
# StripePrivy: delegated consent가 실제 wallet에 적용되었는지 확인
# CoinbaseCDP에서는 건너뜀 — delegation은 wallet별이 아니라 project level에서 구성됨
if CREDENTIAL_PROVIDER_TYPE == "StripePrivy":
    from utils import verify_privy_signer_on_wallet

    try:
        ok = verify_privy_signer_on_wallet(
            app_id=require_env("PRIVY_APP_ID"),
            app_secret=require_env("PRIVY_APP_SECRET"),
            wallet_address_or_id=WALLET_ADDRESS,
            quorum_id=require_env("PRIVY_AUTHORIZATION_ID"),
        )
    except Exception as exc:
        print(f"  ⚠️  Consent check skipped: {exc}")
        print("     This is fine if you haven't chosen Connect agent yet —")
        print("     do it now in the Privy reference frontend and re-run this cell.")
    else:
        if ok:
            print(f"  ✅ Signer access granted on {WALLET_ADDRESS}")
            print("     Consent is in place — ProcessPayment will work in Tutorial 01.")
        else:
            print(f"  ❌ Signer access has NOT been granted on {WALLET_ADDRESS}.")
            print("     Open http://localhost:3000, log in as the same email, choose Connect agent,")
            print("     then re-run this cell. ProcessPayment requires this configuration to succeed.")
else:
    print("  CoinbaseCDP: delegation is handled at the CDP project level — no per-wallet check from here.")

## 7c단계 — Wallet Balance 검증(선택 사항)

session을 생성하기 전에 `GetPaymentInstrumentBalance`를 사용하여 wallet에 USDC가 있는지 확인합니다.

In [ ]:
# wallet balance 확인
chain = "BASE_SEPOLIA" if NETWORK == "ETHEREUM" else "SOLANA_DEVNET"
try:
    balance_resp = dp_client.get_payment_instrument_balance(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=CONNECTOR_ID,
        paymentInstrumentId=INSTRUMENT_ID,
        userId=USER_ID,
        chain=chain,
        token="USDC",
    )
    token_balance = balance_resp.get("tokenBalance", {})
    amount = int(token_balance.get("amount", "0")) / 1_000_000
    print(f"✅ Wallet balance: {amount:.2f} USDC on {chain}")
    if amount == 0:
        print("⚠️  Wallet has no USDC yet. Fund it via the faucet before Step 8.")
except Exception as e:
    print(f"⚠️  Balance check failed: {e}")
    print("   You can still proceed to Step 8 — the session creation will succeed even")
    print("   without a balance check. Verify the wallet is funded as described in Step 7b.")

## 8단계 — Payment Session 생성

시간 제한 payment limit입니다. `value`는 **string**이어야 합니다. `currency`는 `USDC`가 아닌 `USD`입니다. service가 limit 적용을 위해 USDC를 USD로 변환합니다.

In [ ]:
resp = dp_client.create_payment_session(
    paymentManagerArn=MANAGER_ARN,
    userId=USER_ID,
    expiryTimeInMinutes=60,
    limits={"maxSpendAmount": {"value": "1.0", "currency": "USD"}},
    clientToken=client_token(),
)
SESSION_ID = resp["paymentSession"]["paymentSessionId"]
print(f"✅ Session: {SESSION_ID} (budget: $1.00, expiry: 60 min)")

# 즉시 저장
from utils import update_env_file

update_env_file({"SESSION_ID": SESSION_ID})

## 8b단계 — Observability 활성화

Payment Manager의 CloudWatch vended log와 X-Ray trace를 활성화합니다. 

자세한 내용은 [이 가이드](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-payments-metrics.html)를 참조하세요.


In [ ]:
# Payment Manager의 observability 활성화
# 필요한 CloudWatch/X-Ray 권한이 없으면 이 셀에서 자동 provision
from utils import enable_observability

account_id = boto3.client("sts").get_caller_identity()["Account"]
caller_arn = boto3.client("sts").get_caller_identity()["Arn"]

# 필요한 경우 현재 role에 observability 권한 자동 연결
try:
    iam = boto3.client("iam")
    # caller ARN에서 role 이름 추출(assumed-role 및 role 형식 처리)
    if ":assumed-role/" in caller_arn:
        current_role_name = caller_arn.split(":")[-1].split("/")[1]
    elif ":role/" in caller_arn:
        current_role_name = caller_arn.split("/")[-1]
    else:
        current_role_name = None

    if current_role_name:
        obs_policy = {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "CloudWatchLogsVendedDelivery",
                    "Effect": "Allow",
                    "Action": [
                        "logs:CreateDelivery",
                        "logs:CreateLogGroup",
                        "logs:CreateLogStream",
                        "logs:DeleteDelivery",
                        "logs:DeleteDeliveryDestination",
                        "logs:DeleteDeliverySource",
                        "logs:DeleteLogGroup",
                        "logs:DeleteResourcePolicy",
                        "logs:DescribeLogGroups",
                        "logs:DescribeResourcePolicies",
                        "logs:GetDelivery",
                        "logs:GetDeliveryDestination",
                        "logs:GetDeliverySource",
                        "logs:PutDeliveryDestination",
                        "logs:PutDeliverySource",
                        "logs:PutLogEvents",
                        "logs:PutResourcePolicy",
                        "logs:PutRetentionPolicy",
                    ],
                    "Resource": f"arn:aws:logs:{AWS_REGION}:{account_id}:log-group:/aws/vendedlogs/bedrock-agentcore/*",
                },
                {
                    "Sid": "XRayApplicationSignalsCloudTrail",
                    "Effect": "Allow",
                    "Action": [
                        "xray:GetTraceSegmentDestination",
                        "xray:ListResourcePolicies",
                        "xray:PutResourcePolicy",
                        "xray:PutTelemetryRecords",
                        "xray:PutTraceSegments",
                        "xray:UpdateTraceSegmentDestination",
                        "application-signals:StartDiscovery",
                        "cloudtrail:CreateServiceLinkedChannel",
                    ],
                    "Resource": f"arn:aws:xray:{AWS_REGION}:{account_id}:*",
                },
                {
                    "Sid": "CreateServiceLinkedRoleForAppSignals",
                    "Effect": "Allow",
                    "Action": "iam:CreateServiceLinkedRole",
                    "Resource": "arn:*:iam::*:role/aws-service-role/application-signals.cloudwatch.amazonaws.com/AWSServiceRoleForCloudWatchApplicationSignals",
                },
                {
                    "Sid": "BedrockAgentCoreVendedLogDelivery",
                    "Effect": "Allow",
                    "Action": "bedrock-agentcore:AllowVendedLogDeliveryForResource",
                    "Resource": f"arn:aws:bedrock-agentcore:{AWS_REGION}:{account_id}:*",
                },
            ],
        }
        import json as _json

        iam.put_role_policy(
            RoleName=current_role_name,
            PolicyName="AgentCoreObservabilitySetup",
            PolicyDocument=_json.dumps(obs_policy),
        )
        print(f"  Attached observability permissions to {current_role_name}")
        import time

        time.sleep(5)  # IAM propagation 대기
except Exception as perm_err:
    print(f"  Note: Could not auto-attach permissions ({type(perm_err).__name__}). Proceeding anyway.")

try:
    obs_result = enable_observability(
        resource_arn=MANAGER_ARN,
        resource_id=MANAGER_ID,
        account_id=account_id,
        region=AWS_REGION,
        enable_xray_spans=False,
    )
    print(f"\n  Logs: /aws/vendedlogs/bedrock-agentcore/{MANAGER_ID}")
    print("  View traces: CloudWatch console > X-Ray traces > Traces")
except Exception as e:
    print(f"\n  \u26a0\ufe0f Observability setup failed: {e}")
    print("  This is non-blocking — tutorials will still work without observability.")
    print("  To enable later: AgentCore console > Payment Manager > Log deliveries and tracing.")

## 9단계 — 설정 검증

payment session을 다시 읽어 생성되었고 ACTIVE 상태인지 확인합니다.

In [ ]:
# session 검증 — 다시 읽어 field가 생성한 값과 일치하는지 확인
# PaymentSession에는 lifecycle status가 없으며 GetPaymentSession이 성공하면
# expiry 또는 budget에 도달할 때까지 session이 활성 상태임
resp = dp_client.get_payment_session(paymentManagerArn=MANAGER_ARN, paymentSessionId=SESSION_ID, userId=USER_ID)
session = resp["paymentSession"]

assert session["paymentSessionId"] == SESSION_ID, f"sessionId mismatch: {session['paymentSessionId']}"
assert session.get("expiryTimeInMinutes"), "expiryTimeInMinutes missing"

print(f"  paymentSessionId:    {session['paymentSessionId']}")
print(f"  expiryTimeInMinutes: {session['expiryTimeInMinutes']}")
budget = session.get("limits", {}).get("maxSpendAmount", {})
if budget:
    print(f"  budget:              {budget.get('value')} {budget.get('currency')}")
available = session.get("availableLimits", {}).get("maxSpendAmount", {})
if available:
    print(f"  available:           {available.get('value')} {available.get('currency')}")
print("\n✅ Session is ready for ProcessPayment in Tutorial 01.")

## 리소스 정리

> **리소스를 정리하기 전에:** Credential Provider를 삭제하면 wallet credentials가 영구적으로 제거되고 instrument를 삭제하면 wallet address가 제거됩니다. 이 리소스를 사용하는 이후의 모든 튜토리얼을 완료한 후에만 실행하세요.

이 셀은 self-contained 방식으로 구성되어 `.env`에서 resource ID를 load하므로
Notebook 전체를 다시 실행하지 않고도 사용할 수 있습니다. dependency 순서에 따라 리소스를 삭제합니다.

1. Payment Session(data plane)
2. Payment Instrument(data plane)
3. Payment Connector(control plane)
4. Payment Manager(control plane)
5. Credential Provider(control plane)

각 단계는 idempotent하며 이미 삭제된 리소스는 문제없이 건너뜁니다.

> **비용 안내:** 리소스를 정리한 후 IAM console에서 `setup_payment_roles()`로 생성한 IAM role 네 개도 삭제하고, observability를 활성화했다면 CloudWatch log group(`/aws/vendedlogs/bedrock-agentcore/<manager-id>`)도 삭제하세요. cleanup 셀에서 제거되지 않으며 계속 비용이 발생할 수 있습니다.

In [ ]:
# import os, sys, uuid
# sys.path.append('..')
# from dotenv import load_dotenv
# import boto3
# import botocore.exceptions
# from utils import assume_role, client_token

# # ── .env에서 resource ID 로드(독립 실행 가능) ──────────────────
# load_dotenv(override=True)

# AWS_REGION = os.environ.get('AWS_REGION', 'us-west-2')
# PAYMENTS_CP_ENDPOINT = os.environ.get('PAYMENTS_CP_ENDPOINT', f'https://bedrock-agentcore-control.{AWS_REGION}.amazonaws.com')
# PAYMENTS_DP_ENDPOINT = os.environ.get('PAYMENTS_DP_ENDPOINT', f'https://bedrock-agentcore.{AWS_REGION}.amazonaws.com')
# CREDENTIAL_PROVIDER_ENDPOINT = os.environ.get('CREDENTIAL_PROVIDER_ENDPOINT', PAYMENTS_CP_ENDPOINT)

# MANAGER_ARN = os.environ.get('PAYMENT_MANAGER_ARN', '')
# MANAGER_ID = os.environ.get('PAYMENT_MANAGER_ID', '')
# CONNECTOR_ID = os.environ.get('PAYMENT_CONNECTOR_ID', '')
# CREDENTIAL_PROVIDER_ARN = os.environ.get('CREDENTIAL_PROVIDER_ARN', '')
# USER_ID = os.environ.get('USER_ID', 'test-user-001')
# INSTRUMENT_ID = os.environ.get('INSTRUMENT_ID', '')
# SESSION_ID = os.environ.get('SESSION_ID', '')

# # ARN의 마지막 '/' 뒤 segment에서 credential provider 이름 추출
# CRED_PROVIDER_NAME = CREDENTIAL_PROVIDER_ARN.rsplit('/', 1)[-1] if CREDENTIAL_PROVIDER_ARN else ''

# CONTROL_PLANE_ROLE_ARN = os.environ.get('CONTROL_PLANE_ROLE_ARN', f'arn:aws:iam::{boto3.client("sts").get_caller_identity()["Account"]}:role/AgentCorePaymentsControlPlaneRole')
# MANAGEMENT_ROLE_ARN = os.environ.get('MANAGEMENT_ROLE_ARN', f'arn:aws:iam::{boto3.client("sts").get_caller_identity()["Account"]}:role/AgentCorePaymentsManagementRole')

# # ── 정리할 resource가 있는지 검증 ──────────────────────────────
# if not MANAGER_ID:
#     print('❌ No PAYMENT_MANAGER_ID found in .env — nothing to clean up.')
#     print('   Run Tutorial 00 first, or set PAYMENT_MANAGER_ID in .env.')
# else:
#     print(f'🧹 Cleanup targets (from .env):')
#     print(f'   Manager:             {MANAGER_ID}')
#     print(f'   Connector:           {CONNECTOR_ID or "(none)"}')
#     print(f'   Credential Provider: {CRED_PROVIDER_NAME or "(none)"}')
#     print(f'   Instrument:          {INSTRUMENT_ID or "(none)"}')
#     print(f'   Session:             {SESSION_ID or "(none)"}')
#     print()

In [ ]:
# # ── 적절한 role로 client 구성 ──────────────────────────────────
# session = boto3.Session(region_name=AWS_REGION)

# print('Assuming ControlPlaneRole...')
# cp_session = assume_role(session, CONTROL_PLANE_ROLE_ARN, 'cleanup-cp')
# cp_client = cp_session.client('bedrock-agentcore-control', endpoint_url=PAYMENTS_CP_ENDPOINT)
# cred_client = cp_session.client('bedrock-agentcore-control', endpoint_url=CREDENTIAL_PROVIDER_ENDPOINT)

# print('Assuming ManagementRole...')
# mgmt_session = assume_role(session, MANAGEMENT_ROLE_ARN, 'cleanup-mgmt')
# dp_client = mgmt_session.client('bedrock-agentcore', endpoint_url=PAYMENTS_DP_ENDPOINT)

# # ── Helper ─────────────────────────────────────────────────────
# def safe_delete(fn, label, **kw):
#     """삭제 API를 호출하고 ResourceNotFound와 ConflictException을 적절히 처리합니다."""
#     try:
#         fn(**kw)
#         print(f'  ✅ Deleted: {label}')
#     except botocore.exceptions.ClientError as e:
#         code = e.response['Error']['Code']
#         if code in ('ResourceNotFoundException', 'NotFoundException'):
#             print(f'  ⏭️  Already gone: {label}')
#         elif code == 'ConflictException':
#             print(f'  ⚠️  Conflict (may have dependents): {label}')
#             print(f'      {e.response["Error"]["Message"]}')
#         else:
#             raise

# # ── 1. Payment Session 삭제 ────────────────────────────────────
# print('\n── Cleaning up Payment Sessions ──')
# if MANAGER_ARN:
#     try:
#         sessions_resp = dp_client.list_payment_sessions(paymentManagerArn=MANAGER_ARN, userId=USER_ID)
#         sessions = sessions_resp.get('paymentSessions', [])
#         if sessions:
#             for s in sessions:
#                 sid = s.get('paymentSessionId', s.get('sessionId', ''))
#                 if sid:
#                     safe_delete(dp_client.update_payment_session, f'Session {sid} (expire)',
#                         paymentManagerArn=MANAGER_ARN, paymentSessionId=sid, userId=USER_ID,
#                         expiryTimeInMinutes=0)
#             print(f'  Expired {len(sessions)} session(s)')
#         else:
#             print('  No sessions found')
#     except botocore.exceptions.ClientError as e:
#         if 'NotFound' in e.response['Error']['Code']:
#             print('  Manager not found — skipping session cleanup')
#         else:
#             print(f'  ⚠️  Could not list sessions: {e.response["Error"]["Message"]}')
# else:
#     print('  No MANAGER_ARN — skipping')

# # ── 2. Payment Instrument 삭제 ─────────────────────────────────
# print('\n── Cleaning up Payment Instruments ──')
# if MANAGER_ARN:
#     try:
#         instr_resp = dp_client.list_payment_instruments(paymentManagerArn=MANAGER_ARN, userId=USER_ID)
#         instruments = instr_resp.get('paymentInstruments', [])
#         if instruments:
#             for instr in instruments:
#                 iid = instr.get('paymentInstrumentId', instr.get('instrumentId', ''))
#                 if iid:
#                     safe_delete(dp_client.delete_payment_instrument, f'Instrument {iid}',
#                         paymentManagerArn=MANAGER_ARN, paymentInstrumentId=iid, userId=USER_ID)
#         else:
#             print('  No instruments found')
#     except botocore.exceptions.ClientError as e:
#         if 'NotFound' in e.response['Error']['Code']:
#             print('  Manager not found — skipping instrument cleanup')
#         else:
#             print(f'  ⚠️  Could not list instruments: {e.response["Error"]["Message"]}')
# else:
#     print('  No MANAGER_ARN — skipping')

# # ── 3. Payment Connector 삭제 ──────────────────────────────────
# print('\n── Cleaning up Payment Connectors ──')
# if MANAGER_ID:
#     try:
#         conn_resp = cp_client.list_payment_connectors(paymentManagerId=MANAGER_ID)
#         connectors = conn_resp.get('paymentConnectors', [])
#         if connectors:
#             for conn in connectors:
#                 cid = conn.get('paymentConnectorId', conn.get('connectorId', ''))
#                 if cid:
#                     safe_delete(cp_client.delete_payment_connector, f'Connector {cid}',
#                         paymentManagerId=MANAGER_ID, paymentConnectorId=cid, clientToken=client_token())
#         else:
#             print('  No connectors found')
#     except botocore.exceptions.ClientError as e:
#         if 'NotFound' in e.response['Error']['Code']:
#             print('  Manager not found — skipping connector cleanup')
#         else:
#             print(f'  ⚠️  Could not list connectors: {e.response["Error"]["Message"]}')
# else:
#     print('  No MANAGER_ID — skipping')

# # ── 4. Payment Manager 삭제 ────────────────────────────────────
# print('\n── Cleaning up Payment Manager ──')
# if MANAGER_ID:
#     safe_delete(cp_client.delete_payment_manager, f'Manager {MANAGER_ID}',
#         paymentManagerId=MANAGER_ID, clientToken=client_token())
# else:
#     print('  No MANAGER_ID — skipping')

# # ── 5. Credential Provider 삭제 ────────────────────────────────
# print('\n── Cleaning up Credential Provider ──')
# if CRED_PROVIDER_NAME:
#     safe_delete(cred_client.delete_payment_credential_provider, f'CredProvider {CRED_PROVIDER_NAME}',
#         name=CRED_PROVIDER_NAME)
# else:
#     print('  No CREDENTIAL_PROVIDER_ARN — skipping')

# print('\n' + '=' * 60)
# print('  🧹 Cleanup complete')
# print('=' * 60)

# 축하합니다!

첫 번째 payment-enabled agent를 구축하려면 **[Tutorial 01 — Agent에 Payment Limit 활성화](../01-agents-payments-and-limits/)**로 계속 진행하세요.